# 🧠 Notebook 04 — Judge Layer, Confidence & Explainability (Improved)

## Changes from v1

| Problem | Fix |
|---|---|
| Missing scores (0) drag down verdict incorrectly | **Category-adaptive weighting** — weights adjusted by video category |
| Confidence always too high/too low | **Multi-signal confidence** — uses ToT branch divergence + comment count + noise |
| Explanations are generic | **Evidence-grounded explanations** — pulls actual quotes from agent outputs |
| Single threshold (7.5) ignores context | **Context-aware thresholds** — Education scored harder, Music scored softer |
| Radar chart hides N/A scores | **Annotated radar** — N/A dimensions marked differently |
| No trend/category analysis | **Category breakdown chart** — compare across content types |

In [4]:
# ── Cell 1: Imports & Paths ──────────────────────────────────────────
import os, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PROJECT_ROOT = Path(r"C:\Users\user\Documents\LLM - AI\Project\agentic_video_analysis")
DATA_DIR     = PROJECT_ROOT / "data"
PROC_DIR     = DATA_DIR / "processed"
RESULTS_DIR  = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

pq = PROC_DIR / "comments_dataset_clean.parquet"
cv = PROC_DIR / "comments_dataset_clean.csv"
if pq.exists():   df = pd.read_parquet(pq)
elif cv.exists(): df = pd.read_csv(cv)
else: raise FileNotFoundError("Run Notebook 01 first.")
print(f"✅ Dataset: {df.shape}")

OSError: Repetition level histogram size mismatch

In [ ]:
# ── Cell 2: Load Agent Results ───────────────────────────────────────
def load_all_agent_results():
    results = {}
    for p in sorted(RESULTS_DIR.glob("agent_result_*.json")):
        vid_id = p.stem.replace("agent_result_", "")
        with open(p, encoding="utf-8") as f:
            results[vid_id] = json.load(f)
    return results

all_results = load_all_agent_results()
print(f"✅ Loaded {len(all_results)} agent result files.")
if not all_results:
    print("⚠️  No results. Run Notebook 03 first.")

In [2]:
df = pd.read_csv(cv)

In [3]:
df = pd.read_parquet(pq, engine="fastparquet")

ImportError: Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
# ── Cell 3: Category-Adaptive Scoring ───────────────────────────────
#
# KEY IMPROVEMENT: Different content types have different expectations.
# A music video SHOULD have low discourse/info — that's normal, not bad.
# An education video SHOULD have high info — we hold it to a higher standard.
#
# Base weights (general content):
BASE_WEIGHTS = {
    "sentiment":   0.20,
    "noise":       0.15,
    "discourse":   0.25,
    "info":        0.20,
    "helpfulness": 0.20,
}

# Per-category weight overrides
CATEGORY_WEIGHTS = {
    "Music":           {"sentiment":0.40, "noise":0.20, "discourse":0.10, "info":0.10, "helpfulness":0.20},
    "Entertainment":   {"sentiment":0.35, "noise":0.20, "discourse":0.15, "info":0.10, "helpfulness":0.20},
    "Film & Animation":{"sentiment":0.35, "noise":0.20, "discourse":0.20, "info":0.10, "helpfulness":0.15},
    "Gaming":          {"sentiment":0.25, "noise":0.20, "discourse":0.20, "info":0.10, "helpfulness":0.25},
    "Education":       {"sentiment":0.15, "noise":0.10, "discourse":0.25, "info":0.30, "helpfulness":0.20},
    "Science & Technology": {"sentiment":0.15, "noise":0.10, "discourse":0.25, "info":0.30, "helpfulness":0.20},
    "Howto & Style":   {"sentiment":0.15, "noise":0.15, "discourse":0.15, "info":0.20, "helpfulness":0.35},
    "News & Politics": {"sentiment":0.20, "noise":0.15, "discourse":0.30, "info":0.25, "helpfulness":0.10},
    "People & Blogs":  {"sentiment":0.35, "noise":0.20, "discourse":0.20, "info":0.10, "helpfulness":0.15},
    "Comedy":          {"sentiment":0.40, "noise":0.20, "discourse":0.15, "info":0.10, "helpfulness":0.15},
    "Sports":          {"sentiment":0.30, "noise":0.20, "discourse":0.20, "info":0.15, "helpfulness":0.15},
}

# Per-category verdict thresholds (Education is held to higher standard)
CATEGORY_THRESHOLDS = {
    "Music":           {"good": 6.5, "caution": 4.0},
    "Entertainment":   {"good": 6.5, "caution": 4.0},
    "Film & Animation":{"good": 6.5, "caution": 4.0},
    "Comedy":          {"good": 6.5, "caution": 4.0},
    "Gaming":          {"good": 7.0, "caution": 5.0},
    "Education":       {"good": 7.5, "caution": 5.5},
    "Science & Technology": {"good": 7.5, "caution": 5.5},
    "Howto & Style":   {"good": 7.0, "caution": 5.0},
    "News & Politics": {"good": 7.0, "caution": 5.0},
    "default":         {"good": 7.0, "caution": 5.0},
}

_ZERO_THRESHOLD = 0.5  # Below this = N/A, not genuine zero

SCORE_KEYS = {
    "sentiment":   "sentiment_score",
    "noise":       "noise_score",
    "discourse":   "discourse_depth_score",
    "info":        "info_density_score",
    "helpfulness": "helpfulness_score",
}

def safe(d, k, dv=0.0):
    if not isinstance(d, dict): return dv
    try: return float(d.get(k, dv))
    except: return dv


def compute_final_score(result: dict, category: str = ""):
    """
    Compute final score with:
    1. Category-adaptive weights
    2. Missing-score-aware (N/A excluded)
    3. Category-aware verdict thresholds
    """
    weights = CATEGORY_WEIGHTS.get(category, BASE_WEIGHTS)
    thresholds = CATEGORY_THRESHOLDS.get(category, CATEGORY_THRESHOLDS["default"])
    
    raw_scores = {}
    for agent, key in SCORE_KEYS.items():
        s = safe(result.get(agent), key)
        if 0 < s <= 1.0:
            s = s * 10  # auto-scale
        raw_scores[agent] = round(max(0.0, min(10.0, s)), 2)
    
    # Separate available vs N/A
    available = {a: s for a, s in raw_scores.items() if s > _ZERO_THRESHOLD}
    na_dims   = [a for a in raw_scores if raw_scores[a] <= _ZERO_THRESHOLD]
    
    if available:
        weight_sum = sum(weights[a] for a in available)
        score = sum(raw_scores[a] * weights[a] / weight_sum for a in available)
    else:
        vals = list(raw_scores.values())
        score = sum(vals) / len(vals) if vals else 5.0
    
    score = round(max(0.0, min(10.0, score)), 2)
    
    # Context-aware verdict
    if score >= thresholds["good"]:
        verdict = "✅ Worth watching"
    elif score >= thresholds["caution"]:
        verdict = "⚠️ Watch with caution"
    else:
        verdict = "❌ Consider skipping"
    
    return score, verdict, raw_scores, na_dims


# Self-test: music video with no discourse/info
_test = {
    "sentiment":   {"sentiment_score": 7.3},
    "noise":       {"noise_score": 7.3},
    "discourse":   {"discourse_depth_score": 0.0},
    "info":        {"info_density_score": 0.0},
    "helpfulness": {"helpfulness_score": 7.3},
}
_s, _v, _cs, _na = compute_final_score(_test, "Music")
assert _s >= 6.5, f"Music video should score ≥6.5 but got {_s}"
assert "Worth" in _v, f"Expected 'Worth watching' for music but got '{_v}'"
print(f"✅ Category-adaptive scoring verified.")
print(f"   Music video: score={_s}, verdict='{_v}', N/A dims={_na}")

In [ ]:
# ── Cell 4: Multi-Signal Confidence ─────────────────────────────────
#
# OLD: confidence = comment count only
# NEW: 4 signals:
#   1. Comment count (sample size)
#   2. ToT branch divergence (if A and B agreed → higher confidence)
#   3. LLM vs heuristic (LLM used → higher confidence)
#   4. Noise penalty (spammy comments → lower reliability)

def compute_confidence(df_src, vid_id, result, agents_raw):
    n = len(df_src[df_src["video_id"] == str(vid_id)])
    
    # Signal 1: sample size
    if n >= 150: base = 0.92
    elif n >= 100: base = 0.85
    elif n >= 50:  base = 0.72
    elif n >= 20:  base = 0.55
    else:          base = 0.38
    
    # Signal 2: ToT branch agreement (lower divergence = higher confidence)
    divergence_penalty = 0.0
    for agent_key in ["sentiment", "noise", "helpfulness"]:
        agent_data = agents_raw.get(agent_key, {})
        if isinstance(agent_data, dict) and "tot_branch_a_score" in agent_data:
            a_s = agent_data["tot_branch_a_score"]
            b_s = agent_data["tot_branch_b_score"]
            divergence = abs(a_s - b_s) / 10.0  # normalize to 0-1
            divergence_penalty += divergence * 0.05  # max 0.15 penalty across 3 agents
    
    # Signal 3: fallback penalty (heuristics are less reliable than LLM)
    fallback_count = sum(
        1 for a in agents_raw.values()
        if isinstance(a, dict) and "heuristic" in a.get("explanation","")
    )
    fallback_penalty = fallback_count * 0.04  # 4% per agent that fell back
    
    # Signal 4: noise penalty
    noise = result.get("noise", {})
    noise_penalty = (
        0.15 * safe(noise, "off_topic_ratio") +
        0.10 * safe(noise, "spam_ratio")
    ) if isinstance(noise, dict) else 0.0
    
    conf = base - divergence_penalty - fallback_penalty - noise_penalty
    return round(max(0.10, min(0.97, conf)), 2)


print("✅ Multi-signal confidence ready.")

In [ ]:
# ── Cell 5: Evidence-Grounded Explainability ─────────────────────────
#
# OLD: generic template strings ("Comment section is clean")
# NEW: pulls actual quotes from agent outputs for concrete evidence

def build_reasons(component_scores: dict, result: dict, na_dims: list, category: str = ""):
    reasons = []
    
    s  = component_scores.get("sentiment",   0)
    n  = component_scores.get("noise",       0)
    d  = component_scores.get("discourse",   0)
    i  = component_scores.get("info",        0)
    h  = component_scores.get("helpfulness", 0)
    mr = safe(result.get("info"), "misinformation_risk", 0)
    
    # Annotate with category context
    if category:
        reasons.append(f"📂 Content type: **{category}** (scoring weights adjusted accordingly)")
    
    # ── Sentiment ────────────────────────────────────────────────────
    if "sentiment" not in na_dims:
        sent_data = result.get("sentiment", {})
        top_pos = sent_data.get("top_positive", []) if isinstance(sent_data, dict) else []
        top_neg = sent_data.get("top_negative", []) if isinstance(sent_data, dict) else []
        
        if s >= 8:
            quote = f' (e.g. "{top_pos[0][:60]}")' if top_pos else ""
            reasons.append(f"💚 **Strongly positive** audience reaction: {s:.1f}/10{quote}")
        elif s >= 6:
            reasons.append(f"😊 **Mostly positive** audience sentiment: {s:.1f}/10")
        elif s >= 4:
            reasons.append(f"😐 **Mixed** audience sentiment: {s:.1f}/10")
        else:
            quote = f' (e.g. "{top_neg[0][:60]}")' if top_neg else ""
            reasons.append(f"🔴 **Notable dissatisfaction**: {s:.1f}/10{quote}")
    else:
        reasons.append("💬 Sentiment data: insufficient signal for this video.")
    
    # ── Noise ────────────────────────────────────────────────────────
    if "noise" not in na_dims:
        noise_data = result.get("noise", {})
        spam_ratio = safe(noise_data, "spam_ratio")
        flagged    = noise_data.get("flagged_patterns", []) if isinstance(noise_data, dict) else []
        
        if n >= 8:
            reasons.append(f"🧹 **Clean comment section**: {n:.1f}/10 — spam rate {spam_ratio:.0%}")
        elif n >= 6:
            reasons.append(f"⚠️ **Moderate noise**: {n:.1f}/10 — some spam/off-topic comments detected")
        else:
            pattern_note = f" (e.g. '{flagged[0]}')" if flagged else ""
            reasons.append(f"🚨 **Heavy spam/noise**: {n:.1f}/10{pattern_note}")
    else:
        reasons.append("🔇 Noise data: insufficient signal.")
    
    # ── Discourse ────────────────────────────────────────────────────
    if "discourse" not in na_dims:
        disc_data = result.get("discourse", {})
        examples  = disc_data.get("notable_examples", []) if isinstance(disc_data, dict) else []
        if d >= 7:
            quote = f' (e.g. "{examples[0][:70]}")' if examples else ""
            reasons.append(f"💬 **Rich discussion** detected: {d:.1f}/10{quote}")
        elif d >= 5:
            reasons.append(f"💬 **Moderate discussion** depth: {d:.1f}/10")
        else:
            reasons.append(f"💤 **Limited discussion** depth: {d:.1f}/10")
    else:
        cat_note = f" (normal for {category})" if category else ""
        reasons.append(f"ℹ️ Discourse not applicable{cat_note} — excluded from rating.")
    
    # ── Info Quality ─────────────────────────────────────────────────
    if "info" not in na_dims:
        info_data = result.get("info", {})
        experts   = info_data.get("expert_comments", []) if isinstance(info_data, dict) else []
        if i >= 7:
            quote = f' (e.g. "{experts[0][:70]}")' if experts else ""
            reasons.append(f"📚 **Strong information quality**: {i:.1f}/10{quote}")
        elif i >= 5:
            reasons.append(f"📖 **Moderate information quality**: {i:.1f}/10")
        else:
            reasons.append(f"📉 **Low information density**: {i:.1f}/10")
    else:
        cat_note = f" (normal for {category})" if category else ""
        reasons.append(f"ℹ️ Info quality not applicable{cat_note} — excluded from rating.")
    
    # ── Helpfulness ──────────────────────────────────────────────────
    if "helpfulness" not in na_dims:
        help_data = result.get("helpfulness", {})
        tips      = help_data.get("best_helpful_comments", []) if isinstance(help_data, dict) else []
        if h >= 7:
            quote = f' (e.g. "{tips[0][:60]}")' if tips else ""
            reasons.append(f"🛠️ **Highly helpful** comments: {h:.1f}/10{quote}")
        elif h >= 5:
            reasons.append(f"🛠️ **Moderately helpful** comments: {h:.1f}/10")
        else:
            reasons.append(f"❓ **Low practical value**: {h:.1f}/10")
    else:
        cat_note = f" (normal for {category})" if category else ""
        reasons.append(f"ℹ️ Helpfulness not applicable{cat_note} — excluded from rating.")
    
    if mr > 0.5:
        reasons.append(f"🚨 **Misinformation risk** flagged in comments (risk={mr:.0%})")
    
    return reasons


print("✅ Evidence-grounded explainability ready.")

In [ ]:
# ── Cell 6: Generate All Final Reports ──────────────────────────────
all_reports = []

for vid_id, result in all_results.items():
    rows_v   = df[df["video_id"] == vid_id]
    title    = str(rows_v.iloc[0].get("title",   "")) if not rows_v.empty else vid_id
    channel  = str(rows_v.iloc[0].get("channel", "")) if not rows_v.empty else ""
    views    = int(rows_v.iloc[0].get("views",    0)) if not rows_v.empty else 0
    category = str(rows_v.iloc[0].get("category_name", "")) if not rows_v.empty else ""
    
    score, verdict, component_scores, na_dims = compute_final_score(result, category)
    conf    = compute_confidence(df, vid_id, result, result)
    reasons = build_reasons(component_scores, result, na_dims, category)
    
    report = {
        "video_id":         vid_id,
        "title":            title,
        "channel":          channel,
        "category_name":    category,
        "views":            views,
        "final_score":      score,
        "verdict":          verdict,
        "confidence":       conf,
        "comment_count":    len(rows_v),
        "na_dimensions":    na_dims,
        "component_scores": component_scores,
        "reasons":          reasons,
        "sentiment":     result.get("sentiment"),
        "noise":         result.get("noise"),
        "discourse":     result.get("discourse"),
        "info":          result.get("info"),
        "helpfulness":   result.get("helpfulness"),
    }
    
    rpath = RESULTS_DIR / f"final_report_{vid_id}.json"
    with open(rpath, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    all_reports.append(report)

print(f"✅ Generated {len(all_reports)} final reports.")
print("\nTop 10 by score:")
for r in sorted(all_reports, key=lambda x: x["final_score"], reverse=True)[:10]:
    cs = r.get("component_scores", {})
    na = r.get("na_dimensions", [])
    print(f"  {r['final_score']:5.2f}/10  [{int(r['confidence']*100):3d}%conf]  {r['title'][:45]}")
    print(f"         [{r['category_name'][:15]}] sent={cs.get('sentiment',0):.1f} "
          f"noise={cs.get('noise',0):.1f} disc={cs.get('discourse',0):.1f} "
          f"info={cs.get('info',0):.1f} help={cs.get('helpfulness',0):.1f}  N/A:{na}")

In [ ]:
# ── Cell 7: Charts ───────────────────────────────────────────────────
BG = "#FAFAFA"; PANEL = "#FFFFFF"; TEXT = "#1e293b"
GREEN = "#059669"; AMBER = "#d97706"; RED = "#dc2626"

# ── 7a: Leaderboard Bar Chart ─────────────────────────────────────
if all_reports:
    top    = sorted(all_reports, key=lambda r: r["final_score"], reverse=True)[:12]
    titles = [f"{r.get('title','?')[:28]}" for r in top]
    scores = [r["final_score"] for r in top]
    confs  = [r.get("confidence", 0) for r in top]
    cats   = [r.get("category_name","?")[:12] for r in top]
    colors = [GREEN if s >= 7.0 else AMBER if s >= 5 else RED for s in scores]

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor(BG); ax.set_facecolor(PANEL)
    bars = ax.barh(titles[::-1], scores[::-1], color=colors[::-1], height=0.6, edgecolor="none")
    for bar, s, c, cat in zip(bars, scores[::-1], confs[::-1], cats[::-1]):
        ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
                f"{s:.2f}  ({int(c*100)}%) [{cat}]",
                va="center", color=TEXT, fontsize=8.5)
    ax.set_xlim(0, 12)
    ax.axvline(7.0, color=GREEN, ls="--", alpha=0.5, lw=1.5, label="Worth watching")
    ax.axvline(5.0, color=AMBER, ls="--", alpha=0.5, lw=1.5, label="Watch with caution")
    ax.set_xlabel("Quality Score (0–10)", fontsize=11)
    ax.set_title("VideoSense AI — Quality Leaderboard (Category-Adaptive)",
                 fontsize=14, fontweight="bold", loc="left")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "leaderboard_chart.png", dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print("✅ Leaderboard chart saved.")

In [ ]:
# ── 7b: Category Breakdown Chart (NEW) ───────────────────────────────
if len(all_reports) > 1:
    cat_data = {}
    for r in all_reports:
        cat = r.get("category_name", "Unknown") or "Unknown"
        cat_data.setdefault(cat, []).append(r["final_score"])
    
    if len(cat_data) > 1:
        cats   = sorted(cat_data.keys())
        means  = [np.mean(cat_data[c]) for c in cats]
        counts = [len(cat_data[c]) for c in cats]
        colors = [GREEN if m >= 7 else AMBER if m >= 5 else RED for m in means]
        
        fig, ax = plt.subplots(figsize=(10, 4))
        fig.patch.set_facecolor(BG); ax.set_facecolor(PANEL)
        bars = ax.bar(cats, means, color=colors, edgecolor="none", width=0.5)
        for bar, m, cnt in zip(bars, means, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f"{m:.1f}\n(n={cnt})", ha="center", color=TEXT, fontsize=9)
        ax.set_ylim(0, 11)
        ax.set_ylabel("Avg Score (0-10)")
        ax.set_title("Average Quality Score by Category", fontsize=13, fontweight="bold", loc="left")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "category_breakdown.png", dpi=150, bbox_inches="tight", facecolor=BG)
        plt.show()
        print("✅ Category breakdown chart saved.")

In [ ]:
# ── 7c: Radar chart for best video ───────────────────────────────────
def plot_radar(report):
    cs  = report.get("component_scores", {})
    na  = report.get("na_dimensions", [])
    cats = ["Sentiment", "Low Noise", "Discourse", "Info Quality", "Helpfulness"]
    keys = ["sentiment", "noise", "discourse", "info", "helpfulness"]
    vals = [cs.get(k, 0) / 10 for k in keys]
    
    N      = len(cats)
    angles = [n/N*2*np.pi for n in range(N)]
    angles += angles[:1]; vals += vals[:1]
    
    fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw=dict(polar=True))
    ax.plot(angles, vals, "o-", lw=2.5, color="#0d9488")
    ax.fill(angles, vals, alpha=0.18, color="#0d9488")
    
    # Mark N/A dimensions with grey
    for i, (cat, key) in enumerate(zip(cats, keys)):
        if key in na:
            cats[i] = cats[i] + "\n(N/A)"
    
    ax.set_thetagrids(np.degrees(angles[:-1]), cats, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([.25,.5,.75,1])
    ax.set_yticklabels(["2.5","5","7.5","10"], fontsize=7, color="gray")
    plt.title(
        f"{report['title'][:45]}\n"
        f"Score: {report['final_score']}/10 | {report['verdict']} | Conf: {int(report['confidence']*100)}%",
        fontsize=9, y=1.12
    )
    plt.tight_layout()
    plt.show()


if all_reports:
    best = max(all_reports, key=lambda r: r["final_score"])
    print(f"Best video: {best['title'][:60]}")
    print(f"Category: {best['category_name']}")
    print(f"Score: {best['final_score']}/10 | {best['verdict']} | Confidence: {int(best['confidence']*100)}%")
    print("\nReasons:")
    for reason in best.get("reasons", []):
        print(f"  {reason}")
    plot_radar(best)

In [ ]:
# ── Cell 8: Summary & Leaderboard CSV ────────────────────────────────
if all_reports:
    worth   = sum(1 for r in all_reports if "Worth" in r["verdict"])
    caution = sum(1 for r in all_reports if "caution" in r["verdict"])
    skip    = sum(1 for r in all_reports if "skipping" in r["verdict"])
    total   = len(all_reports)
    avg_conf = sum(r["confidence"] for r in all_reports) / total
    
    print(f"{'='*55}")
    print(f"  FINAL SUMMARY — {total} videos analyzed")
    print(f"{'='*55}")
    print(f"  ✅ Worth watching    : {worth:3d} ({100*worth//max(total,1)}%)")
    print(f"  ⚠️  Watch with caution: {caution:3d} ({100*caution//max(total,1)}%)")
    print(f"  ❌ Consider skipping : {skip:3d} ({100*skip//max(total,1)}%)")
    print(f"  📊 Avg confidence    : {int(avg_conf*100)}%")
    print(f"{'='*55}")
    print("\n✅ Notebook 04 complete. → Now run 05_demo_interface_improved.py")

lb_rows = [{
    "title":      r.get("title","?")[:40],
    "channel":    r.get("channel","?")[:20],
    "category":   r.get("category_name","?")[:15],
    "score":      r["final_score"],
    "verdict":    r["verdict"],
    "confidence": r["confidence"],
    "n_comments": r.get("comment_count", 0),
    "na_dims":    ",".join(r.get("na_dimensions", [])),
} for r in all_reports]

lb = pd.DataFrame(lb_rows).sort_values("score", ascending=False).reset_index(drop=True)
lb.index += 1
lb.to_csv(RESULTS_DIR / "leaderboard.csv", index=False)
print("\n📊 Leaderboard:")
print(lb.to_string())